In [ ]:
import sys
from pathlib import Path

sys.path.append(str(Path.cwd()))

import pickle

import numpy as np
import pandas as pd
import statsmodels.api as sm
from linearmodels.panel import PanelOLS, RandomEffects, PooledOLS
from scipy.stats import chi2, f
from statsmodels.stats.outliers_influence import variance_inflation_factor

from Modules.panel_utils import (
    ModelResultsAggregator,
    run_panel_regressions,
    run_spec_tests,
    run_panel_model_diagnostics,
)
from Modules.test_fun import *


In [ ]:
###############################
# ЗАГРУЗКА И ПОДГОТОВКА ДАННЫХ
###############################
df_reg_analys = pd.read_excel('Operations/fl_cred_reg_analys.xlsx')
df_fed_analys = pd.read_excel('Operations/fed_analys.xlsx')

# Убираем служебные столбцы индекса
df_reg_analys = df_reg_analys.loc[:, ~df_reg_analys.columns.str.startswith('Unnamed')]
df_fed_analys = df_fed_analys.loc[:, ~df_fed_analys.columns.str.startswith('Unnamed')]

# Приводим даты к datetime
df_reg_analys['Date'] = pd.to_datetime(df_reg_analys['Date'])
df_fed_analys['Date'] = pd.to_datetime(df_fed_analys['Date'])
 

cols_to_merge = ['CAR_Indicator', 'Bonds_Rate_Correct_5Y',
                'Covid_dum', 'Sank_dum', 'Oil_p', 'CPI', 'REER',
                'd_ROISFIX', 'd_MIACR',] # 'd_IBC_constr_fed_adj', 'd_IBC_torg_fed_adj', 'd_IBC_auto_fed_adj',

# Объединяем региональные и федеральные данные
fed_extra_cols = [
    col for col in df_fed_analys.columns
    if col not in df_reg_analys.columns and col in cols_to_merge
]
df_reg = df_reg_analys.merge(
    df_fed_analys[['Date'] + fed_extra_cols],
    on='Date',
    how='left'
)


df_reg = df_reg.sort_values(['Region', 'Date']).copy()

df_reg['Cluster_new_cd_rest'] = ((df_reg['Cluster_new_cd_1'] == 0) & (df_reg['Cluster_new_cd_3'] == 0) & (df_reg['Cluster_new_cd_4'] == 0)).astype(int)  


#  Создаем взаимодействия для переменных процентных ставок
map_vars_int = ['d_Mon_Shock_neg', 'd_Mon_Shock_pos',
                'd_ROISFIX_neg', 'd_ROISFIX_pos',
                'd_MIACR_neg','d_MIACR_pos',
]

z_vars_int = [
    'Covid_dum',
    'Sank_dum',
    'Cluster_new_cd_1',
    'Cluster_new_cd_3',
    'Cluster_new_cd_4'
]

z_cols_int = list()
for m in map_vars_int:
    for z in z_vars_int:
        col_name = f"{m}_{z}"
        z_cols_int.append(col_name)
        df_reg[col_name] = df_reg[m] * df_reg[z]



In [ ]:
df_reg.columns

# Модель

### Создание модели

In [ ]:
# Сортировка
df_reg = df_reg.sort_values(['Region', 'Date']).copy()

# Переменные и их лаги
lag1_map = [
    "d_Int_Rate_FL", "d_ln_New_Loans_Fl", 
    "Zadolg_ConsCred", "Zadolg_Fl", "Def_Zadolg_ConsCred", "Def_Zadolg_Fl",
    "Zakred", "Cap_to_assets", "CAR_Indicator", "CPI", "Inflation_Expectations_adj",
    'CPI_reg', 'Cred_nagr',
]

# Лаги переменных 
for lag_col in lag1_map:
    if lag_col in df_reg.columns:
        df_reg[f"{lag_col}_lag1"] = df_reg.groupby('Region')[lag_col].shift(1)

# Лаги шоков
for shock_col in ['d_Mon_Shock_pos', 'd_Mon_Shock_neg', 
                  'd_ROISFIX_pos', 'd_ROISFIX_neg',
                  'd_MIACR_pos', 'd_MIACR_neg',
                  'd_Mon_Shock_neg_Covid_dum', 'd_Mon_Shock_neg_Sank_dum',
                  'd_Mon_Shock_pos_Covid_dum', 'd_Mon_Shock_pos_Sank_dum',
                  'd_ROISFIX_neg_Covid_dum', 'd_ROISFIX_neg_Sank_dum', 
                  'd_ROISFIX_pos_Covid_dum', 'd_ROISFIX_pos_Sank_dum',
                  'd_MIACR_neg_Covid_dum', 'd_MIACR_neg_Sank_dum', 
                  'd_MIACR_pos_Covid_dum', 'd_MIACR_pos_Sank_dum',
                  'd_Mon_Shock_neg_Cluster_new_cd_1', 'd_Mon_Shock_neg_Cluster_new_cd_3', 'd_Mon_Shock_neg_Cluster_new_cd_4',
                  'd_Mon_Shock_pos_Cluster_new_cd_1', 'd_Mon_Shock_pos_Cluster_new_cd_3', 'd_Mon_Shock_pos_Cluster_new_cd_4',
                  'd_ROISFIX_neg_Cluster_new_cd_1', 'd_ROISFIX_neg_Cluster_new_cd_3', 'd_ROISFIX_neg_Cluster_new_cd_4',
                  'd_ROISFIX_pos_Cluster_new_cd_1', 'd_ROISFIX_pos_Cluster_new_cd_3', 'd_ROISFIX_pos_Cluster_new_cd_4',
                  'd_MIACR_neg_Cluster_new_cd_1', 'd_MIACR_neg_Cluster_new_cd_3', 'd_MIACR_neg_Cluster_new_cd_4',
                  'd_MIACR_pos_Cluster_new_cd_1', 'd_MIACR_pos_Cluster_new_cd_3', 'd_MIACR_pos_Cluster_new_cd_4']:      
    if shock_col in df_reg.columns:
        for k in range(1, 13):
            df_reg[f"{shock_col}_lag{k}"] = df_reg.groupby('Region')[shock_col].shift(k)



####################
# Кластерные выборки
####################
if 'Cluster_new_cd_1' in df_reg.columns:
    df_reg_clus_one = df_reg[df_reg['Cluster_new_cd_1'] == 1].copy()
    df_reg_clus_three = df_reg[df_reg['Cluster_new_cd_3'] == 1].copy()
    df_reg_clus_four = df_reg[df_reg['Cluster_new_cd_4'] == 1].copy()
    df_reg_clus_rest = df_reg[df_reg['Cluster_new_cd_rest'] == 1].copy()

In [ ]:
ogranich = False  # Убираем период до СВО

if (ogranich == True):
    df_reg = df_reg.query("Date >= '01.01.2023'").reset_index(drop=True)

# Все модели


In [ ]:
###############################
# Создание аргументов модели
###############################

dependent_var = 'd_Int_Rate_FL'

exog_vars_base = [
    'd_Int_Rate_FL_lag1',             # Лаг зависимой
    # 'd_ln_New_Loans_Fl',              # Показатели портфеля
    'd_ln_New_Loans_Fl_lag1'
    # 'Zadolg_ConsCred_lag1',
    # 'Def_Zadolg_Fl'
    'Def_Zadolg_Fl_lag1',     
    'Cred_nagr_lag1',                    # Закредитованность / Кредитная нагрузка (Станислав за закредитованность)
    'Zakred_lag1',                        
    'ln_Fin_Dostup',                           # Доля фин орг / Доля топ-5
    # 'D_top5_rozn',                  
    # 'Bonds_Rate_Correct_5Y',                # Ставка по облигациям
    'Cap_to_assets_lag1',
    'Oil_p'                                   # Нефть
    'REER',                                   # Валютный курс 
    # 'CPI',                                  # Инфляция
    # 'CPI_lag1',
    # 'd_CPI_lag1',
    # 'CPI_reg',
    'CPI_reg_lag1',      
    
    # 'Inflation_Expectations',               # Инфл ожидания
    # 'Inflation_Expectations_adj',
    # 'Inflation_Expectations_adj_lag1',
    'd_Inflation_Expectations',     
    'Cluster_new_cd_1',
    'Cluster_new_cd_3',
    'Cluster_new_cd_4'
]

# Дамми
if (ogranich == False):
    exog_vars_base.append('Covid_dum')
    exog_vars_base.append('Sank_dum')

shock_vars = [
    ['d_Mon_Shock_pos', 'd_Mon_Shock_neg'],
    ['d_Mon_Shock_pos_lag1', 'd_Mon_Shock_neg_lag1'],
    ['d_Mon_Shock_pos_lag2', 'd_Mon_Shock_neg_lag2'],  
    ['d_Mon_Shock_pos_lag3', 'd_Mon_Shock_neg_lag3'],
    ['d_Mon_Shock_pos_lag5', 'd_Mon_Shock_neg_lag5'],
    ['d_Mon_Shock_pos_lag6', 'd_Mon_Shock_neg_lag6'],
    ['d_Mon_Shock_pos_lag7', 'd_Mon_Shock_neg_lag7'],
    ['d_Mon_Shock_pos_lag8', 'd_Mon_Shock_neg_lag8'],
    ['d_Mon_Shock_pos_lag9', 'd_Mon_Shock_neg_lag9'],
    ['d_Mon_Shock_pos_lag10', 'd_Mon_Shock_neg_lag10'],
    ['d_Mon_Shock_pos_lag11', 'd_Mon_Shock_neg_lag11'],
    ['d_Mon_Shock_pos_lag12', 'd_Mon_Shock_neg_lag12'],
    ['d_Mon_Shock_pos', 'd_Mon_Shock_neg', 'd_Mon_Shock_neg_Covid_dum', 'd_Mon_Shock_neg_Sank_dum', 'd_Mon_Shock_pos_Covid_dum', 'd_Mon_Shock_pos_Sank_dum'],
    ['d_Mon_Shock_pos_lag1', 'd_Mon_Shock_neg_lag1', 'd_Mon_Shock_neg_Covid_dum_lag1', 'd_Mon_Shock_neg_Sank_dum_lag1', 'd_Mon_Shock_pos_Covid_dum_lag1', 'd_Mon_Shock_pos_Sank_dum_lag1',],
    ['d_Mon_Shock_pos_lag2', 'd_Mon_Shock_neg_lag2', 'd_Mon_Shock_neg_Covid_dum_lag2', 'd_Mon_Shock_neg_Sank_dum_lag2', 'd_Mon_Shock_pos_Covid_dum_lag2', 'd_Mon_Shock_pos_Sank_dum_lag2',],  
    ['d_Mon_Shock_pos_lag3', 'd_Mon_Shock_neg_lag3', 'd_Mon_Shock_neg_Covid_dum_lag3', 'd_Mon_Shock_neg_Sank_dum_lag3', 'd_Mon_Shock_pos_Covid_dum_lag3', 'd_Mon_Shock_pos_Sank_dum_lag3',],
    ['d_Mon_Shock_pos_lag4', 'd_Mon_Shock_neg_lag4', 'd_Mon_Shock_neg_Covid_dum_lag4', 'd_Mon_Shock_neg_Sank_dum_lag4', 'd_Mon_Shock_pos_Covid_dum_lag4', 'd_Mon_Shock_pos_Sank_dum_lag4',],
    ['d_Mon_Shock_pos_lag5', 'd_Mon_Shock_neg_lag5', 'd_Mon_Shock_neg_Covid_dum_lag5', 'd_Mon_Shock_neg_Sank_dum_lag5', 'd_Mon_Shock_pos_Covid_dum_lag5', 'd_Mon_Shock_pos_Sank_dum_lag5',],
    ['d_Mon_Shock_pos_lag6', 'd_Mon_Shock_neg_lag6', 'd_Mon_Shock_neg_Covid_dum_lag6', 'd_Mon_Shock_neg_Sank_dum_lag6', 'd_Mon_Shock_pos_Covid_dum_lag6', 'd_Mon_Shock_pos_Sank_dum_lag6',],
    ['d_Mon_Shock_pos_lag7', 'd_Mon_Shock_neg_lag7', 'd_Mon_Shock_neg_Covid_dum_lag7', 'd_Mon_Shock_neg_Sank_dum_lag7', 'd_Mon_Shock_pos_Covid_dum_lag7', 'd_Mon_Shock_pos_Sank_dum_lag7',],
    ['d_Mon_Shock_pos_lag8', 'd_Mon_Shock_neg_lag8', 'd_Mon_Shock_neg_Covid_dum_lag8', 'd_Mon_Shock_neg_Sank_dum_lag8', 'd_Mon_Shock_pos_Covid_dum_lag8', 'd_Mon_Shock_pos_Sank_dum_lag8',],
    ['d_Mon_Shock_pos_lag9', 'd_Mon_Shock_neg_lag9', 'd_Mon_Shock_neg_Covid_dum_lag9', 'd_Mon_Shock_neg_Sank_dum_lag9', 'd_Mon_Shock_pos_Covid_dum_lag9', 'd_Mon_Shock_pos_Sank_dum_lag9',],
    ['d_Mon_Shock_pos_lag10', 'd_Mon_Shock_neg_lag10', 'd_Mon_Shock_neg_Covid_dum_lag10', 'd_Mon_Shock_neg_Sank_dum_lag10', 'd_Mon_Shock_pos_Covid_dum_lag10', 'd_Mon_Shock_pos_Sank_dum_lag10',],
    ['d_Mon_Shock_pos_lag11', 'd_Mon_Shock_neg_lag11', 'd_Mon_Shock_neg_Covid_dum_lag11', 'd_Mon_Shock_neg_Sank_dum_lag11', 'd_Mon_Shock_pos_Covid_dum_lag11', 'd_Mon_Shock_pos_Sank_dum_lag11',],
    ['d_Mon_Shock_pos_lag12', 'd_Mon_Shock_neg_lag12', 'd_Mon_Shock_neg_Covid_dum_lag12', 'd_Mon_Shock_neg_Sank_dum_lag12', 'd_Mon_Shock_pos_Covid_dum_lag12', 'd_Mon_Shock_pos_Sank_dum_lag12',],
    ['d_Mon_Shock_neg', 'd_Mon_Shock_neg_Cluster_new_cd_1', 'd_Mon_Shock_neg_Cluster_new_cd_3','d_Mon_Shock_neg_Cluster_new_cd_4',   #
    'd_Mon_Shock_pos', 'd_Mon_Shock_pos_Cluster_new_cd_1', 'd_Mon_Shock_pos_Cluster_new_cd_3','d_Mon_Shock_pos_Cluster_new_cd_4',],  #
    ['d_Mon_Shock_neg_lag1', 'd_Mon_Shock_neg_Cluster_new_cd_1_lag1', 'd_Mon_Shock_neg_Cluster_new_cd_3_lag1','d_Mon_Shock_neg_Cluster_new_cd_4_lag1',   #
    'd_Mon_Shock_pos_lag1', 'd_Mon_Shock_pos_Cluster_new_cd_1_lag1', 'd_Mon_Shock_pos_Cluster_new_cd_3_lag1','d_Mon_Shock_pos_Cluster_new_cd_4_lag1',],  #
    ['d_Mon_Shock_neg_lag2', 'd_Mon_Shock_neg_Cluster_new_cd_1_lag2', 'd_Mon_Shock_neg_Cluster_new_cd_3_lag2','d_Mon_Shock_neg_Cluster_new_cd_4_lag2',   #
    'd_Mon_Shock_pos_lag2', 'd_Mon_Shock_pos_Cluster_new_cd_1_lag2', 'd_Mon_Shock_pos_Cluster_new_cd_3_lag2','d_Mon_Shock_pos_Cluster_new_cd_4_lag2',],  #
    ['d_Mon_Shock_neg_lag3', 'd_Mon_Shock_neg_Cluster_new_cd_1_lag3', 'd_Mon_Shock_neg_Cluster_new_cd_3_lag3','d_Mon_Shock_neg_Cluster_new_cd_4_lag3',   #
    'd_Mon_Shock_pos_lag3', 'd_Mon_Shock_pos_Cluster_new_cd_1_lag3', 'd_Mon_Shock_pos_Cluster_new_cd_3_lag3','d_Mon_Shock_pos_Cluster_new_cd_4_lag3',],  #
    ['d_Mon_Shock_neg_lag4', 'd_Mon_Shock_neg_Cluster_new_cd_1_lag4', 'd_Mon_Shock_neg_Cluster_new_cd_3_lag4','d_Mon_Shock_neg_Cluster_new_cd_4_lag4',   #
    'd_Mon_Shock_pos_lag4', 'd_Mon_Shock_pos_Cluster_new_cd_1_lag4', 'd_Mon_Shock_pos_Cluster_new_cd_3_lag4','d_Mon_Shock_pos_Cluster_new_cd_4_lag4',],  #
    ['d_Mon_Shock_neg_lag5', 'd_Mon_Shock_neg_Cluster_new_cd_1_lag5', 'd_Mon_Shock_neg_Cluster_new_cd_3_lag5','d_Mon_Shock_neg_Cluster_new_cd_4_lag5',   #
    'd_Mon_Shock_pos_lag5', 'd_Mon_Shock_pos_Cluster_new_cd_1_lag5', 'd_Mon_Shock_pos_Cluster_new_cd_3_lag5','d_Mon_Shock_pos_Cluster_new_cd_4_lag5',],  #
    ['d_Mon_Shock_neg_lag6', 'd_Mon_Shock_neg_Cluster_new_cd_1_lag6', 'd_Mon_Shock_neg_Cluster_new_cd_3_lag6', 'd_Mon_Shock_neg_Cluster_new_cd_4_lag6',  #
    'd_Mon_Shock_pos_lag6', 'd_Mon_Shock_pos_Cluster_new_cd_1_lag6', 'd_Mon_Shock_pos_Cluster_new_cd_3_lag6','d_Mon_Shock_pos_Cluster_new_cd_4_lag6',],  #
    ['d_Mon_Shock_neg_lag7', 'd_Mon_Shock_neg_Cluster_new_cd_1_lag7', 'd_Mon_Shock_neg_Cluster_new_cd_3_lag7','d_Mon_Shock_neg_Cluster_new_cd_4_lag7',
    'd_Mon_Shock_pos_lag7', 'd_Mon_Shock_pos_Cluster_new_cd_1_lag7', 'd_Mon_Shock_pos_Cluster_new_cd_3_lag7','d_Mon_Shock_pos_Cluster_new_cd_4_lag7',],  #
    ['d_Mon_Shock_neg_lag8', 'd_Mon_Shock_neg_Cluster_new_cd_1_lag8', 'd_Mon_Shock_neg_Cluster_new_cd_3_lag8','d_Mon_Shock_neg_Cluster_new_cd_4_lag8',
    'd_Mon_Shock_pos_lag8', 'd_Mon_Shock_pos_Cluster_new_cd_1_lag8', 'd_Mon_Shock_pos_Cluster_new_cd_3_lag8','d_Mon_Shock_pos_Cluster_new_cd_4_lag8',],  #
    ['d_Mon_Shock_neg_lag9', 'd_Mon_Shock_neg_Cluster_new_cd_1_lag9', 'd_Mon_Shock_neg_Cluster_new_cd_3_lag9','d_Mon_Shock_neg_Cluster_new_cd_4_lag9',
    'd_Mon_Shock_pos_lag9', 'd_Mon_Shock_pos_Cluster_new_cd_1_lag9', 'd_Mon_Shock_pos_Cluster_new_cd_3_lag9','d_Mon_Shock_pos_Cluster_new_cd_4_lag9',],  #
    ['d_Mon_Shock_neg_lag10', 'd_Mon_Shock_neg_Cluster_new_cd_1_lag10', 'd_Mon_Shock_neg_Cluster_new_cd_3_lag10','d_Mon_Shock_neg_Cluster_new_cd_4_lag10',
    'd_Mon_Shock_pos_lag10', 'd_Mon_Shock_pos_Cluster_new_cd_1_lag10', 'd_Mon_Shock_pos_Cluster_new_cd_3_lag10','d_Mon_Shock_pos_Cluster_new_cd_4_lag10',],  #
    ['d_Mon_Shock_neg_lag11', 'd_Mon_Shock_neg_Cluster_new_cd_1_lag11', 'd_Mon_Shock_neg_Cluster_new_cd_3_lag11','d_Mon_Shock_neg_Cluster_new_cd_4_lag11',
    'd_Mon_Shock_pos_lag11', 'd_Mon_Shock_pos_Cluster_new_cd_1_lag11', 'd_Mon_Shock_pos_Cluster_new_cd_3_lag11','d_Mon_Shock_pos_Cluster_new_cd_4_lag11',],  #
    ['d_Mon_Shock_neg_lag12', 'd_Mon_Shock_neg_Cluster_new_cd_1_lag12', 'd_Mon_Shock_neg_Cluster_new_cd_3_lag12','d_Mon_Shock_neg_Cluster_new_cd_4_lag12',
    'd_Mon_Shock_pos_lag12', 'd_Mon_Shock_pos_Cluster_new_cd_1_lag12', 'd_Mon_Shock_pos_Cluster_new_cd_3_lag12','d_Mon_Shock_pos_Cluster_new_cd_4_lag12',],  #
    

    ['d_ROISFIX_pos', 'd_ROISFIX_neg'],
    ['d_ROISFIX_pos_lag1', 'd_ROISFIX_neg_lag1'],
    ['d_ROISFIX_pos_lag2', 'd_ROISFIX_neg_lag2'],
    ['d_ROISFIX_pos_lag3', 'd_ROISFIX_neg_lag3'],
    ['d_ROISFIX_pos_lag4', 'd_ROISFIX_neg_lag4'],
    ['d_ROISFIX_pos_lag5', 'd_ROISFIX_neg_lag5'],
    ['d_ROISFIX_pos_lag6', 'd_ROISFIX_neg_lag6'],
    ['d_ROISFIX_pos_lag7', 'd_ROISFIX_neg_lag7'],
    ['d_ROISFIX_pos_lag8', 'd_ROISFIX_neg_lag8'],
    ['d_ROISFIX_pos_lag9', 'd_ROISFIX_neg_lag9'],
    ['d_ROISFIX_pos_lag10', 'd_ROISFIX_neg_lag10'],
    ['d_ROISFIX_pos_lag11', 'd_ROISFIX_neg_lag11'],
    ['d_ROISFIX_pos_lag12', 'd_ROISFIX_neg_lag12'],
    ['d_ROISFIX_pos', 'd_ROISFIX_neg', 'd_ROISFIX_neg_Covid_dum', 'd_ROISFIX_neg_Sank_dum','d_ROISFIX_pos_Covid_dum', 'd_ROISFIX_pos_Sank_dum'],
    ['d_ROISFIX_pos_lag1', 'd_ROISFIX_neg_lag1', 'd_ROISFIX_neg_Covid_dum_lag1', 'd_ROISFIX_neg_Sank_dum_lag1', 'd_ROISFIX_pos_Covid_dum_lag1', 'd_ROISFIX_pos_Sank_dum_lag1',],
    ['d_ROISFIX_pos_lag2', 'd_ROISFIX_neg_lag2', 'd_ROISFIX_neg_Covid_dum_lag2', 'd_ROISFIX_neg_Sank_dum_lag2', 'd_ROISFIX_pos_Covid_dum_lag2', 'd_ROISFIX_pos_Sank_dum_lag2',],
    ['d_ROISFIX_pos_lag3', 'd_ROISFIX_neg_lag3', 'd_ROISFIX_neg_Covid_dum_lag3', 'd_ROISFIX_neg_Sank_dum_lag3', 'd_ROISFIX_pos_Covid_dum_lag3', 'd_ROISFIX_pos_Sank_dum_lag3',],
    ['d_ROISFIX_pos_lag4', 'd_ROISFIX_neg_lag4', 'd_ROISFIX_neg_Covid_dum_lag4', 'd_ROISFIX_neg_Sank_dum_lag4', 'd_ROISFIX_pos_Covid_dum_lag4', 'd_ROISFIX_pos_Sank_dum_lag4',],
    ['d_ROISFIX_pos_lag5', 'd_ROISFIX_neg_lag5', 'd_ROISFIX_neg_Covid_dum_lag5', 'd_ROISFIX_neg_Sank_dum_lag5', 'd_ROISFIX_pos_Covid_dum_lag5', 'd_ROISFIX_pos_Sank_dum_lag5',],
    ['d_ROISFIX_pos_lag6', 'd_ROISFIX_neg_lag6', 'd_ROISFIX_neg_Covid_dum_lag6', 'd_ROISFIX_neg_Sank_dum_lag6', 'd_ROISFIX_pos_Covid_dum_lag6', 'd_ROISFIX_pos_Sank_dum_lag6',],
    ['d_ROISFIX_pos_lag7', 'd_ROISFIX_neg_lag7', 'd_ROISFIX_neg_Covid_dum_lag7', 'd_ROISFIX_neg_Sank_dum_lag7', 'd_ROISFIX_pos_Covid_dum_lag7', 'd_ROISFIX_pos_Sank_dum_lag7',],
    ['d_ROISFIX_pos_lag8', 'd_ROISFIX_neg_lag8', 'd_ROISFIX_neg_Covid_dum_lag8', 'd_ROISFIX_neg_Sank_dum_lag8', 'd_ROISFIX_pos_Covid_dum_lag8', 'd_ROISFIX_pos_Sank_dum_lag8',],
    ['d_ROISFIX_pos_lag9', 'd_ROISFIX_neg_lag9', 'd_ROISFIX_neg_Covid_dum_lag9', 'd_ROISFIX_neg_Sank_dum_lag9', 'd_ROISFIX_pos_Covid_dum_lag9', 'd_ROISFIX_pos_Sank_dum_lag9',],
    ['d_ROISFIX_pos_lag10', 'd_ROISFIX_neg_lag10', 'd_ROISFIX_neg_Covid_dum_lag10', 'd_ROISFIX_neg_Sank_dum_lag10', 'd_ROISFIX_pos_Covid_dum_lag10', 'd_ROISFIX_pos_Sank_dum_lag10',],
    ['d_ROISFIX_pos_lag11', 'd_ROISFIX_neg_lag11', 'd_ROISFIX_neg_Covid_dum_lag11', 'd_ROISFIX_neg_Sank_dum_lag11', 'd_ROISFIX_pos_Covid_dum_lag11', 'd_ROISFIX_pos_Sank_dum_lag11',],
    ['d_ROISFIX_pos_lag12', 'd_ROISFIX_neg_lag12', 'd_ROISFIX_neg_Covid_dum_lag12', 'd_ROISFIX_neg_Sank_dum_lag12', 'd_ROISFIX_pos_Covid_dum_lag12', 'd_ROISFIX_pos_Sank_dum_lag12',],
    ['d_ROISFIX_neg', 'd_ROISFIX_neg_Cluster_new_cd_1', 'd_ROISFIX_neg_Cluster_new_cd_3','d_ROISFIX_neg_Cluster_new_cd_4',   #
    'd_ROISFIX_pos', 'd_ROISFIX_pos_Cluster_new_cd_1', 'd_ROISFIX_pos_Cluster_new_cd_3','d_ROISFIX_pos_Cluster_new_cd_4',],  #
    ['d_ROISFIX_neg_lag1', 'd_ROISFIX_neg_Cluster_new_cd_1_lag1', 'd_ROISFIX_neg_Cluster_new_cd_3_lag1','d_ROISFIX_neg_Cluster_new_cd_4_lag1',
    'd_ROISFIX_pos_lag1', 'd_ROISFIX_pos_Cluster_new_cd_1_lag1', 'd_ROISFIX_pos_Cluster_new_cd_3_lag1','d_ROISFIX_pos_Cluster_new_cd_4_lag1',],  #
    ['d_ROISFIX_neg_lag2', 'd_ROISFIX_neg_Cluster_new_cd_1_lag2', 'd_ROISFIX_neg_Cluster_new_cd_3_lag2','d_ROISFIX_neg_Cluster_new_cd_4_lag2',
    'd_ROISFIX_pos_lag2', 'd_ROISFIX_pos_Cluster_new_cd_1_lag2', 'd_ROISFIX_pos_Cluster_new_cd_3_lag2','d_ROISFIX_pos_Cluster_new_cd_4_lag2',],  #
    ['d_ROISFIX_neg_lag3', 'd_ROISFIX_neg_Cluster_new_cd_1_lag3', 'd_ROISFIX_neg_Cluster_new_cd_3_lag3','d_ROISFIX_neg_Cluster_new_cd_4_lag3',
    'd_ROISFIX_pos_lag3', 'd_ROISFIX_pos_Cluster_new_cd_1_lag3', 'd_ROISFIX_pos_Cluster_new_cd_3_lag3','d_ROISFIX_pos_Cluster_new_cd_4_lag3',],  #
    ['d_ROISFIX_neg_lag4', 'd_ROISFIX_neg_Cluster_new_cd_1_lag4', 'd_ROISFIX_neg_Cluster_new_cd_3_lag4','d_ROISFIX_neg_Cluster_new_cd_4_lag4',
    'd_ROISFIX_pos_lag4', 'd_ROISFIX_pos_Cluster_new_cd_1_lag4', 'd_ROISFIX_pos_Cluster_new_cd_3_lag4','d_ROISFIX_pos_Cluster_new_cd_4_lag4',],  #
    ['d_ROISFIX_neg_lag5', 'd_ROISFIX_neg_Cluster_new_cd_1_lag5', 'd_ROISFIX_neg_Cluster_new_cd_3_lag5', 'd_ROISFIX_neg_Cluster_new_cd_4_lag5',
    'd_ROISFIX_pos_lag5', 'd_ROISFIX_pos_Cluster_new_cd_1_lag5', 'd_ROISFIX_pos_Cluster_new_cd_3_lag5','d_ROISFIX_pos_Cluster_new_cd_4_lag5',],  #
    ['d_ROISFIX_neg_lag6', 'd_ROISFIX_neg_Cluster_new_cd_1_lag6', 'd_ROISFIX_neg_Cluster_new_cd_3_lag6','d_ROISFIX_neg_Cluster_new_cd_4_lag6',
    'd_ROISFIX_pos_lag6', 'd_ROISFIX_pos_Cluster_new_cd_1_lag6', 'd_ROISFIX_pos_Cluster_new_cd_3_lag6','d_ROISFIX_pos_Cluster_new_cd_4_lag6',],  #
    ['d_ROISFIX_neg_lag7', 'd_ROISFIX_neg_Cluster_new_cd_1_lag7', 'd_ROISFIX_neg_Cluster_new_cd_3_lag7','d_ROISFIX_neg_Cluster_new_cd_4_lag7',
    'd_ROISFIX_pos_lag7', 'd_ROISFIX_pos_Cluster_new_cd_1_lag7', 'd_ROISFIX_pos_Cluster_new_cd_3_lag7','d_ROISFIX_pos_Cluster_new_cd_4_lag7',],  #
    ['d_ROISFIX_neg_lag8', 'd_ROISFIX_neg_Cluster_new_cd_1_lag8', 'd_ROISFIX_neg_Cluster_new_cd_3_lag8','d_ROISFIX_neg_Cluster_new_cd_4_lag8',
    'd_ROISFIX_pos_lag8', 'd_ROISFIX_pos_Cluster_new_cd_1_lag8', 'd_ROISFIX_pos_Cluster_new_cd_3_lag8','d_ROISFIX_pos_Cluster_new_cd_4_lag8',],  #
    ['d_ROISFIX_neg_lag9', 'd_ROISFIX_neg_Cluster_new_cd_1_lag9', 'd_ROISFIX_neg_Cluster_new_cd_3_lag9','d_ROISFIX_neg_Cluster_new_cd_4_lag9',
    'd_ROISFIX_pos_lag9', 'd_ROISFIX_pos_Cluster_new_cd_1_lag9', 'd_ROISFIX_pos_Cluster_new_cd_3_lag9','d_ROISFIX_pos_Cluster_new_cd_4_lag9',],  #
    ['d_ROISFIX_neg_lag10', 'd_ROISFIX_neg_Cluster_new_cd_1_lag10', 'd_ROISFIX_neg_Cluster_new_cd_3_lag10','d_ROISFIX_neg_Cluster_new_cd_4_lag10',
    'd_ROISFIX_pos_lag10', 'd_ROISFIX_pos_Cluster_new_cd_1_lag10', 'd_ROISFIX_pos_Cluster_new_cd_3_lag10','d_ROISFIX_pos_Cluster_new_cd_4_lag10',],  #
    ['d_ROISFIX_neg_lag11', 'd_ROISFIX_neg_Cluster_new_cd_1_lag11', 'd_ROISFIX_neg_Cluster_new_cd_3_lag11','d_ROISFIX_neg_Cluster_new_cd_4_lag11',
    'd_ROISFIX_pos_lag11', 'd_ROISFIX_pos_Cluster_new_cd_1_lag11', 'd_ROISFIX_pos_Cluster_new_cd_3_lag11','d_ROISFIX_pos_Cluster_new_cd_4_lag11',],  #
    ['d_ROISFIX_neg_lag12', 'd_ROISFIX_neg_Cluster_new_cd_1_lag12', 'd_ROISFIX_neg_Cluster_new_cd_3_lag12','d_ROISFIX_neg_Cluster_new_cd_4_lag12',
    'd_ROISFIX_pos_lag12', 'd_ROISFIX_pos_Cluster_new_cd_1_lag12', 'd_ROISFIX_pos_Cluster_new_cd_3_lag12','d_ROISFIX_pos_Cluster_new_cd_4_lag12',],  #


    ['d_MIACR_pos', 'd_MIACR_neg'],
    ['d_MIACR_pos_lag1', 'd_MIACR_neg_lag1'],
    ['d_MIACR_pos_lag2', 'd_MIACR_neg_lag2'],
    ['d_MIACR_pos_lag3', 'd_MIACR_neg_lag3'],
    ['d_MIACR_pos_lag4', 'd_MIACR_neg_lag4'],
    ['d_MIACR_pos_lag5', 'd_MIACR_neg_lag5'],
    ['d_MIACR_pos_lag6', 'd_MIACR_neg_lag6'],
    ['d_MIACR_pos_lag7', 'd_MIACR_neg_lag7'],
    ['d_MIACR_pos_lag8', 'd_MIACR_neg_lag8'],
    ['d_MIACR_pos_lag9', 'd_MIACR_neg_lag9'],
    ['d_MIACR_pos_lag10', 'd_MIACR_neg_lag10'],
    ['d_MIACR_pos_lag11', 'd_MIACR_neg_lag11'],
    ['d_MIACR_pos_lag12', 'd_MIACR_neg_lag12'],
    ['d_MIACR_pos', 'd_MIACR_neg', 'd_MIACR_neg_Covid_dum', 'd_MIACR_neg_Sank_dum', 'd_MIACR_pos_Covid_dum', 'd_MIACR_pos_Sank_dum'],
    ['d_MIACR_pos_lag1', 'd_MIACR_neg_lag1', 'd_MIACR_neg_Covid_dum_lag1', 'd_MIACR_neg_Sank_dum_lag1', 'd_MIACR_pos_Covid_dum_lag1', 'd_MIACR_pos_Sank_dum_lag1',],
    ['d_MIACR_pos_lag2', 'd_MIACR_neg_lag2', 'd_MIACR_neg_Covid_dum_lag2', 'd_MIACR_neg_Sank_dum_lag2', 'd_MIACR_pos_Covid_dum_lag2', 'd_MIACR_pos_Sank_dum_lag2',],
    ['d_MIACR_pos_lag3', 'd_MIACR_neg_lag3', 'd_MIACR_neg_Covid_dum_lag3', 'd_MIACR_neg_Sank_dum_lag3', 'd_MIACR_pos_Covid_dum_lag3', 'd_MIACR_pos_Sank_dum_lag3',],
    ['d_MIACR_pos_lag4', 'd_MIACR_neg_lag4', 'd_MIACR_neg_Covid_dum_lag4', 'd_MIACR_neg_Sank_dum_lag4', 'd_MIACR_pos_Covid_dum_lag4', 'd_MIACR_pos_Sank_dum_lag4',],
    ['d_MIACR_pos_lag5', 'd_MIACR_neg_lag5', 'd_MIACR_neg_Covid_dum_lag5', 'd_MIACR_neg_Sank_dum_lag5', 'd_MIACR_pos_Covid_dum_lag5', 'd_MIACR_pos_Sank_dum_lag5',],
    ['d_MIACR_pos_lag6', 'd_MIACR_neg_lag6', 'd_MIACR_neg_Covid_dum_lag6', 'd_MIACR_neg_Sank_dum_lag6', 'd_MIACR_pos_Covid_dum_lag6', 'd_MIACR_pos_Sank_dum_lag6'],
    ['d_MIACR_pos_lag7', 'd_MIACR_neg_lag7', 'd_MIACR_neg_Covid_dum_lag7', 'd_MIACR_neg_Sank_dum_lag7', 'd_MIACR_pos_Covid_dum_lag7', 'd_MIACR_pos_Sank_dum_lag7',],
    ['d_MIACR_pos_lag8', 'd_MIACR_neg_lag8', 'd_MIACR_neg_Covid_dum_lag8', 'd_MIACR_neg_Sank_dum_lag8', 'd_MIACR_pos_Covid_dum_lag8', 'd_MIACR_pos_Sank_dum_lag8',],
    ['d_MIACR_pos_lag9', 'd_MIACR_neg_lag9', 'd_MIACR_neg_Covid_dum_lag9', 'd_MIACR_neg_Sank_dum_lag9', 'd_MIACR_pos_Covid_dum_lag9', 'd_MIACR_pos_Sank_dum_lag9',],
    ['d_MIACR_pos_lag10', 'd_MIACR_neg_lag10', 'd_MIACR_neg_Covid_dum_lag10', 'd_MIACR_neg_Sank_dum_lag10', 'd_MIACR_pos_Covid_dum_lag10', 'd_MIACR_pos_Sank_dum_lag10',],
    ['d_MIACR_pos_lag11', 'd_MIACR_neg_lag11', 'd_MIACR_neg_Covid_dum_lag11', 'd_MIACR_neg_Sank_dum_lag11', 'd_MIACR_pos_Covid_dum_lag11', 'd_MIACR_pos_Sank_dum_lag11',],
    ['d_MIACR_pos_lag12', 'd_MIACR_neg_lag12', 'd_MIACR_neg_Covid_dum_lag12', 'd_MIACR_neg_Sank_dum_lag12', 'd_MIACR_pos_Covid_dum_lag12', 'd_MIACR_pos_Sank_dum_lag12',],
    ['d_MIACR_neg', 'd_MIACR_neg_Cluster_new_cd_1', 'd_MIACR_neg_Cluster_new_cd_3','d_MIACR_neg_Cluster_new_cd_4',
    'd_MIACR_pos', 'd_MIACR_pos_Cluster_new_cd_1', 'd_MIACR_pos_Cluster_new_cd_3','d_MIACR_pos_Cluster_new_cd_4',],  #
    ['d_MIACR_neg_lag1', 'd_MIACR_neg_Cluster_new_cd_1_lag1', 'd_MIACR_neg_Cluster_new_cd_3_lag1','d_MIACR_neg_Cluster_new_cd_4_lag1',
    'd_MIACR_pos_lag1', 'd_MIACR_pos_Cluster_new_cd_1_lag1', 'd_MIACR_pos_Cluster_new_cd_3_lag1', 'd_MIACR_pos_Cluster_new_cd_4_lag1',],  #
    ['d_MIACR_neg_lag2', 'd_MIACR_neg_Cluster_new_cd_1_lag2', 'd_MIACR_neg_Cluster_new_cd_3_lag2','d_MIACR_neg_Cluster_new_cd_4_lag2',
    'd_MIACR_pos_lag2', 'd_MIACR_pos_Cluster_new_cd_1_lag2', 'd_MIACR_pos_Cluster_new_cd_3_lag2','d_MIACR_pos_Cluster_new_cd_4_lag2',],  #
    ['d_MIACR_neg_lag3', 'd_MIACR_neg_Cluster_new_cd_1_lag3', 'd_MIACR_neg_Cluster_new_cd_3_lag3', 'd_MIACR_neg_Cluster_new_cd_4_lag3',
    'd_MIACR_pos_lag3', 'd_MIACR_pos_Cluster_new_cd_1_lag3', 'd_MIACR_pos_Cluster_new_cd_3_lag3','d_MIACR_pos_Cluster_new_cd_4_lag3',],  #
    ['d_MIACR_neg_lag4', 'd_MIACR_neg_Cluster_new_cd_1_lag4', 'd_MIACR_neg_Cluster_new_cd_3_lag4', 'd_MIACR_neg_Cluster_new_cd_4_lag4',
    'd_MIACR_pos_lag4', 'd_MIACR_pos_Cluster_new_cd_1_lag4', 'd_MIACR_pos_Cluster_new_cd_3_lag4','d_MIACR_pos_Cluster_new_cd_4_lag4',],  #
    ['d_MIACR_neg_lag5', 'd_MIACR_neg_Cluster_new_cd_1_lag5', 'd_MIACR_neg_Cluster_new_cd_3_lag5', 'd_MIACR_neg_Cluster_new_cd_4_lag5',
    'd_MIACR_pos_lag5', 'd_MIACR_pos_Cluster_new_cd_1_lag5', 'd_MIACR_pos_Cluster_new_cd_3_lag5','d_MIACR_pos_Cluster_new_cd_4_lag5',],  #
    ['d_MIACR_neg_lag6', 'd_MIACR_neg_Cluster_new_cd_1_lag6', 'd_MIACR_neg_Cluster_new_cd_3_lag6', 'd_MIACR_neg_Cluster_new_cd_4_lag6',
    'd_MIACR_pos_lag6', 'd_MIACR_pos_Cluster_new_cd_1_lag6', 'd_MIACR_pos_Cluster_new_cd_3_lag6','d_MIACR_pos_Cluster_new_cd_4_lag6',],  #
    ['d_MIACR_neg_lag7', 'd_MIACR_neg_Cluster_new_cd_1_lag7', 'd_MIACR_neg_Cluster_new_cd_3_lag7','d_MIACR_neg_Cluster_new_cd_4_lag7',
    'd_MIACR_pos_lag7', 'd_MIACR_pos_Cluster_new_cd_1_lag7', 'd_MIACR_pos_Cluster_new_cd_3_lag7','d_MIACR_pos_Cluster_new_cd_4_lag7',],  #
    ['d_MIACR_neg_lag8', 'd_MIACR_neg_Cluster_new_cd_1_lag8', 'd_MIACR_neg_Cluster_new_cd_3_lag8','d_MIACR_neg_Cluster_new_cd_4_lag8',
    'd_MIACR_pos_lag8', 'd_MIACR_pos_Cluster_new_cd_1_lag8', 'd_MIACR_pos_Cluster_new_cd_3_lag8','d_MIACR_pos_Cluster_new_cd_4_lag8',],  #
    ['d_MIACR_neg_lag9', 'd_MIACR_neg_Cluster_new_cd_1_lag9', 'd_MIACR_neg_Cluster_new_cd_3_lag9','d_MIACR_neg_Cluster_new_cd_4_lag9',
    'd_MIACR_pos_lag9', 'd_MIACR_pos_Cluster_new_cd_1_lag9', 'd_MIACR_pos_Cluster_new_cd_3_lag9','d_MIACR_pos_Cluster_new_cd_4_lag9',],  #
    ['d_MIACR_neg_lag10', 'd_MIACR_neg_Cluster_new_cd_1_lag10', 'd_MIACR_neg_Cluster_new_cd_3_lag10','d_MIACR_neg_Cluster_new_cd_4_lag10',
    'd_MIACR_pos_lag10', 'd_MIACR_pos_Cluster_new_cd_1_lag10', 'd_MIACR_pos_Cluster_new_cd_3_lag10','d_MIACR_pos_Cluster_new_cd_4_lag10',],  #
    ['d_MIACR_neg_lag11', 'd_MIACR_neg_Cluster_new_cd_1_lag11', 'd_MIACR_neg_Cluster_new_cd_3_lag11','d_MIACR_neg_Cluster_new_cd_4_lag11',
    'd_MIACR_pos_lag11', 'd_MIACR_pos_Cluster_new_cd_1_lag11', 'd_MIACR_pos_Cluster_new_cd_3_lag11','d_MIACR_pos_Cluster_new_cd_4_lag11',],  #
    ['d_MIACR_neg_lag12', 'd_MIACR_neg_Cluster_new_cd_1_lag12', 'd_MIACR_neg_Cluster_new_cd_3_lag12','d_MIACR_neg_Cluster_new_cd_4_lag12',
    'd_MIACR_pos_lag12', 'd_MIACR_pos_Cluster_new_cd_1_lag12', 'd_MIACR_pos_Cluster_new_cd_3_lag12','d_MIACR_pos_Cluster_new_cd_4_lag12',],  #
]


###############################
# Автоматическое формирование моделей по shock_vars
###############################

model_spec_all = build_shock_variants(dependent_var, exog_vars_base, shock_vars)
exog_variants = model_spec_all["exog_variants"]

In [ ]:
###############################
# Прогон всех моделей и сохранение результатов
###############################

model_results = {}

for idx, exog_vars_initial in enumerate(exog_variants, 1):
    y, X, pooled_res, fe_res, re_res, pooled_success, fe_success, re_success = run_panel_regressions(
        df_reg, df_reg, dependent_var, exog_vars_initial, cov_type='dk', cluster_entity=True, df_name='df_reg'
    )

    model_results[f"Модель {idx} - OLS"] = pooled_res if pooled_success else None
    model_results[f"Модель {idx} - FE"] = fe_res if fe_success else None
    model_results[f"Модель {idx} - RE"] = re_res if re_success else None


In [ ]:
###############################
# Прогон всех моделей + тесты + экспорт
###############################

from Modules.panel_utils import collect_all_test_pvalues, run_spec_tests_robust
from Modules.model_results_export import ModelResultsAggregator, ensure_results_dir, add_model_set, build_and_export_by_shock_category_with_tests
from Modules.test_fun import detect_shock_type, detect_shock_lag, detect_shock_is_interaction
import os

model_spec_all = build_shock_variants(dependent_var, exog_vars_base, shock_vars)
exog_variants = model_spec_all["exog_variants"]

model_specs_all = []
tests_by_column = {}

for idx, exog_vars_initial in enumerate(exog_variants, 1):
    y, X, pooled_res, fe_res, re_res, pooled_success, fe_success, re_success = run_panel_regressions(
        df_reg, df_reg, dependent_var, exog_vars_initial, cov_type='dk', cluster_entity=True, df_name='df_reg'
    )

    spec_name = f"Модель {idx}"
    shock_entry = shock_vars[idx - 1]
    model_specs_all.append({
        'spec_name': spec_name,
        'dependent_var': dependent_var,
        'subsample': 'Общая выборка',
        'results': {
            'pooled': pooled_res if pooled_success else None,
            'fe': fe_res if fe_success else None,
            're': re_res if re_success else None
        },
        'shock_group':    detect_shock_type(shock_entry),
        'lag_num':        detect_shock_lag(shock_entry),
        'is_interaction': detect_shock_is_interaction(shock_entry),
    })

    tests = collect_all_test_pvalues(
        y, X, pooled_res, fe_res, re_res,
        pooled_success, fe_success, re_success,
        shock_vars=shock_vars
    )
    hausman_stat_r, hausman_pval_r, bp_lm_stat_r, bp_lm_pval_r, f_stat_r, f_pval_r = run_spec_tests_robust(
        y, X, pooled_res, fe_res, re_res,
        pooled_success, fe_success, re_success
    )
    spec_tests_robust = {
        "Hausman (FE vs RE) p-value (robust)": hausman_pval_r,
        "Breusch-Pagan LM (RE vs Pooled) p-value (robust)": bp_lm_pval_r,
        "F-test (FE vs Pooled) p-value (robust)": f_pval_r,
    }

    spec_tests = tests.get('spec_tests', {})
    spec_tests.update(spec_tests_robust)
    diag_tests = tests.get('diagnostics', {})

    for model_type in ['POOL', 'FE', 'RE']:
        col_name = f"{spec_name} ({model_type})"
        col_tests = {}
        for name, pval in spec_tests.items():
            col_tests[name] = pval
        for name, pval in diag_tests.get(model_type, {}).items():
            col_tests[name] = pval
        tests_by_column[col_name] = col_tests

aggregator_all = ModelResultsAggregator()
for spec in model_specs_all:
    add_model_set(aggregator_all, spec)

shock_spec_meta = [
    {k: s[k] for k in ('spec_name', 'shock_group', 'lag_num', 'is_interaction')}
    for s in model_specs_all
]

results_dir = ensure_results_dir('Results')
date_tag = pd.Timestamp.today().strftime('%d_%m_%y')
out_all = os.path.join(results_dir, f"{'d_Int_Rate_Fl'[-2:]}_all_models_with_tests_{date_tag}.xlsx")
build_and_export_by_shock_category_with_tests(
    aggregator_all, shock_spec_meta, tests_by_column, out_all,
    include_pvalues=True, decimals=3, test_decimals=6,
    hausman_key='Hausman (FE vs RE) p-value (robust)'
)

# Подробное изучение

In [ ]:
# df_reg['Region'].unique()

In [ ]:
# ###############################
# # Создание аргументов модели
# ###############################

# dependent_var = 'd_Int_Rate_FL'

# exog_vars_base = [
#     'd_Int_Rate_FL_lag1',             # Лаг зависимой
#     'd_ln_New_Loans_Fl',              # Показатели портфеля
#     # 'd_ln_New_Loans_Fl_lag1'
#     # 'Zadolg_ConsCred_lag1',
#     # 'Def_Zadolg_Fl'
#     'Def_Zadolg_Fl_lag1', 
#     # 'Cred_nagr_lag1',                    # Закредитованность / Кредитная нагрузка (Станислав за закредитованность)
#     'Zakred_lag1',                        
#     'ln_Fin_Dostup',                           # Доля фин орг / Доля топ-5
#     # 'D_top5_rozn',                  
#     # 'CAR_Indicator_lag1',                   # CAR / Капитал к активам / Ставка по облигациям
#     'Cap_to_assets_lag1',
#     # 'Bonds_Rate_Correct_5Y',                # Ставка по облигациям
#     # 'Exc_rate',                             # Валютный курс
#     # 'd_Ex_Rate',
#     'REER',   
#     # 'CPI',                                  # Инфляция
#     # 'CPI_lag1',
#     # 'd_CPI_lag1', 
#     # 'CPI_reg',
#     'CPI_reg_lag1',
       
    
#     # 'Inflation_Expectations',               # Инфл ожидания
#     # 'Inflation_Expectations_adj',
#     # 'Inflation_Expectations_adj_lag1',
#     'd_Inflation_Expectations_adj',     
#     'Covid_dum',                            # Дамми
#     'Sank_dum',
#     'Cluster_new_cd_1',
#     'Cluster_new_cd_3',
#     'Cluster_new_cd_4'
    
#     # Перемножения дамми
# ]
# shock_vars = [
#     # ['d_Mon_Shock_pos', 'd_Mon_Shock_neg'],
#     # ['d_Mon_Shock_pos_lag1', 'd_Mon_Shock_neg_lag1'],
#     # ['d_Mon_Shock_pos_lag2', 'd_Mon_Shock_neg_lag2'],  
#     ['d_Mon_Shock_pos_lag3', 'd_Mon_Shock_neg_lag3'],
#     # ['d_Mon_Shock_pos_lag4', 'd_Mon_Shock_neg_lag4'],
#     # ['d_Mon_Shock_pos_lag5', 'd_Mon_Shock_neg_lag5'],
#     # ['d_Mon_Shock_pos_lag6', 'd_Mon_Shock_neg_lag6'],
#     # ['d_Mon_Shock_pos', 'd_Mon_Shock_pos_Covid_dum', 'd_Mon_Shock_pos_Sank_dum',
#     # 'd_Mon_Shock_neg', 'd_Mon_Shock_neg_Covid_dum', 'd_Mon_Shock_neg_Sank_dum'],

#     ['d_ROISFIX_pos', 'd_ROISFIX_neg'],
#     # ['d_ROISFIX_pos_lag1', 'd_ROISFIX_neg_lag1'],
#     # ['d_ROISFIX_pos_lag1', 'd_ROISFIX_neg_lag1'],
#     # ['d_ROISFIX_pos_lag2', 'd_ROISFIX_neg_lag2'],
#     # ['d_ROISFIX_pos_lag3', 'd_ROISFIX_neg_lag3'],
#     # ['d_ROISFIX_pos_lag4', 'd_ROISFIX_neg_lag4'],
#     # ['d_ROISFIX_pos_lag5', 'd_ROISFIX_neg_lag5'],
#     # ['d_ROISFIX_pos_lag6', 'd_ROISFIX_neg_lag6'],
#     ['d_MIACR_pos', 'd_MIACR_neg'],
#     # ['d_MIACR_pos_lag1', 'd_MIACR_neg_lag1'],
#     # ['d_MIACR_pos_lag2', 'd_MIACR_neg_lag2'],
#     # ['d_MIACR_pos_lag3', 'd_MIACR_neg_lag3'],
#     # ['d_MIACR_pos_lag4', 'd_MIACR_neg_lag4'],
#     # ['d_MIACR_pos_lag5', 'd_MIACR_neg_lag5'],
#     # ['d_MIACR_pos_lag6', 'd_MIACR_neg_lag6'],
# ]

# model_spec = build_shock_variants(dependent_var, exog_vars_base, shock_vars)
# exog_vars_small_1, exog_vars_small_2, exog_vars_small_3 = model_spec["exog_variants"]


### Мультиколлинеарность

In [ ]:
# ###############################
#     #VIF-анализ - начальные панельные данные спецификация 1
# ###############################

# # Tips to colinearity:
# # 'Bonds_Rate_Correct_5Y' with 'CAR_Indicator'
# # 'Bonds_Rate_Correct_5Y' and 'CAR_Indicator' with 'ROISFIX' and 'MIACR'
# # 'Cred_nagr' with 'Zakred'
# # 'Zadolg_ConsCred' with 'New_Loans_ConsCred'

# vif_variables = [item for item in exog_vars_small_1 if item not in ['Cred_nagr','CAR_Indicator','Zadolg_ConsCred']]
# df_vif = df_reg[vif_variables].copy()
# df_vif = df_vif.dropna()

# X_vif = sm.add_constant(df_vif[vif_variables])

# # Расчет VIF
# vif_data = pd.DataFrame()
# vif_data["Variable"] = vif_variables
# vif_data["VIF"] = [variance_inflation_factor(X_vif.values, i+1) for i in range(len(vif_variables))]
# vif_data = vif_data.sort_values('VIF', ascending=False)

# print("\n" + "="*60)
# print("РЕЗУЛЬТАТЫ VIF АНАЛИЗА - НАЧАЛЬНЫЕ ПАНЕЛЬНЫЕ ДАННЫЕ СПЕЦИФИКАЦИЯ 1")
# print("="*60)
# print(vif_data.to_string(index=False))

# exog_vars_small_1 = [item for item in exog_vars_small_1 if item not in ['Cred_nagr','CAR_Indicator','Zadolg_ConsCred']]
# exog_vars_small_2 = [item for item in exog_vars_small_2 if item not in ['Cred_nagr','CAR_Indicator','Zadolg_ConsCred', 'Bonds_Rate_Correct_5Y']]
# exog_vars_small_3 = [item for item in exog_vars_small_3 if item not in ['Cred_nagr','CAR_Indicator','Zadolg_ConsCred', 'Bonds_Rate_Correct_5Y']]

In [ ]:
# ###############################
# # Построение линейных моделей на панельных данных (КАК В ИССЛЕДОВАНИИ СКУРАТОВА & ЗВЕРЕВА)
# # (Int_Rate_ConsCred зависимая переменная)
# # Модель в первой разности - первый лаг ДКП
# ###############################

# dependent_var = model_spec["dependent_var"]
# exog_vars_initial = exog_vars_small_1

# y, X, pooled_res, fe_res, re_res, pooled_success, fe_success, re_success = run_panel_regressions(
#     df_reg, df_reg, dependent_var, exog_vars_initial, cov_type='dk', df_name='df_reg',
# )

# # Сохраняем результаты модели для экспорта
# pooled_res_1 = pooled_res if pooled_success else None
# fe_res_1 = fe_res if fe_success else None
# re_res_1 = re_res if re_success else None


In [ ]:
# run_panel_model_diagnostics(
#     y, X, pooled_res, fe_res, re_res,
#     pooled_success, fe_success, re_success
# )


In [ ]:
# from Modules.panel_utils import run_spec_tests_robust

# run_spec_tests_robust(
#     y, X, pooled_res, fe_res, re_res, pooled_success, fe_success, re_success
# )

In [ ]:
# # ===== ТЕСТЫ СПЕЦИФИКАЦИИ =====

# run_spec_tests(
#     y, X, pooled_res, fe_res, re_res, pooled_success, fe_success, re_success
# )


In [ ]:
# ###############################
# # Построение линейных моделей на панельных данных (КАК В ИССЛЕДОВАНИИ СКУРАТОВА & ЗВЕРЕВА)
# # (Int_Rate_ConsCred зависимая переменная)
# # Модель в первой разности - первый лаг ДКП
# ###############################

# dependent_var = model_spec["dependent_var"]
# exog_vars_initial = exog_vars_small_2

# y, X, pooled_res, fe_res, re_res, pooled_success, fe_success, re_success = run_panel_regressions(
#     df_reg, df_reg, dependent_var, exog_vars_initial, cov_type='dk', cluster_entity=True, df_name='df_reg'
# )

# # Сохраняем результаты модели для экспорта
# pooled_res_2 = pooled_res if pooled_success else None
# fe_res_2 = fe_res if fe_success else None
# re_res_2 = re_res if re_success else None


In [ ]:
# ###############################
# # Построение линейных моделей на панельных данных (КАК В ИССЛЕДОВАНИИ СКУРАТОВА & ЗВЕРЕВА)
# # (Int_Rate_ConsCred зависимая переменная)
# # Модель в первой разности - первый лаг ДКП
# ###############################

# dependent_var = model_spec["dependent_var"]
# exog_vars_initial = exog_vars_small_3

# y, X, pooled_res, fe_res, re_res, pooled_success, fe_success, re_success = run_panel_regressions(
#     df_reg, df_reg, dependent_var, exog_vars_initial, cov_type='dk', cluster_entity=True, df_name='df_reg', 
# )

# # Сохраняем результаты модели для экспорта
# pooled_res_3 = pooled_res if pooled_success else None
# fe_res_3 = fe_res if fe_success else None
# re_res_3 = re_res if re_success else None


In [ ]:
# # Optional: save model specs to Models.pkl
# save_model_spec(dependent_var, exog_vars_base, shock_vars, model_name="Модель 1")
# save_model_spec(dependent_var, exog_vars_base, shock_vars, model_name="Модель 2")
# save_model_spec(dependent_var, exog_vars_base, shock_vars, model_name="Модель 3")


### Вывод результатов

In [ ]:
from Modules.model_results_export import ModelResultsAggregator, ensure_results_dir, add_model_set, build_and_export
import os

dep_var_name = 'Int_Rate_ConsCred'
base_name = dep_var_name
if base_name.startswith(''):
    base_name = base_name[2:]
if base_name.endswith(''):
    base_name = base_name[:-4]

results_dir = ensure_results_dir('Results')

model_specs_all = [
    {
        'spec_name': 'Модель(м) шок Тейлора',
        'dependent_var': dep_var_name,
        'subsample': 'Общая выборка',
        'results': {
            'pooled': pooled_res_1,
            'fe': fe_res_1,
            're': re_res_1
        }
    },
    {
        'spec_name': 'Модель(м) ROISFIX',
        'dependent_var': dep_var_name,
        'subsample': 'Общая выборка',
        'results': {
            'pooled': pooled_res_2,
            'fe': fe_res_2,
            're': re_res_2
        }
    },
    {
        'spec_name': 'Модель(м) MIACR',
        'dependent_var': dep_var_name,
        'subsample': 'Общая выборка',
        'results': {
            'pooled': pooled_res_3,
            'fe': fe_res_3,
            're': re_res_3
        }
    },
]

aggregator_all = ModelResultsAggregator()
for spec in model_specs_all:
    add_model_set(aggregator_all, spec)

out_all = os.path.join(results_dir, f"{base_name}_test.xlsx")
build_and_export(aggregator_all, out_all, include_pvalues=True, decimals=3)

### Выгрузка всех моделей


In [ ]:
# models_dict = load_models()
# models_table = pd.DataFrame.from_dict(models_dict, orient='index')
# models_table


In [ ]:
# from Modules.model_results_export import ModelResultsAggregator, ensure_results_dir, add_model_set, build_and_export
# import os

# models_dict = load_models()

# model_specs_all = []
# for model_name, saved_spec in models_dict.items():
#     model_spec = build_shock_variants(
#         saved_spec["dependent_var"],
#         saved_spec["exog_vars_base"],
#         saved_spec["shock_vars"],
#     )
#     exog_variants = model_spec["exog_variants"]

#     for idx, exog_vars_initial in enumerate(exog_variants):
#         shock_entry = saved_spec["shock_vars"][idx]
#         if isinstance(shock_entry, (list, tuple)):
#             shock_name = "+".join(shock_entry)
#         else:
#             shock_name = str(shock_entry)

#         dependent_var = model_spec["dependent_var"]
#         y, X, pooled_res, fe_res, re_res, pooled_success, fe_success, re_success = run_panel_regressions(
#             df_reg, df_reg, dependent_var, exog_vars_initial, cov_type='clustered', cluster_entity=True, df_name='df_reg'
#         )

#         model_specs_all.append({
#             'spec_name': f"{model_name} - {shock_name}",
#             'dependent_var': dependent_var,
#             'subsample': 'Общая выборка',
#             'results': {
#                 'pooled': pooled_res if pooled_success else None,
#                 'fe': fe_res if fe_success else None,
#                 're': re_res if re_success else None
#             }
#         })

# aggregator_all = ModelResultsAggregator()
# for spec in model_specs_all:
#     add_model_set(aggregator_all, spec)

# results_dir = ensure_results_dir('Results')
# date_tag = pd.Timestamp.today().strftime('%d_%m_%y')
# out_all = os.path.join(results_dir, f"all_models_{date_tag}.xlsx")
# build_and_export(aggregator_all, out_all, include_pvalues=True, decimals=3)


In [ ]:
###############################
# Автоматическое формирование моделей по shock_vars
###############################

model_spec_all = build_shock_variants(dependent_var, exog_vars_base, shock_vars)
exog_variants = model_spec_all["exog_variants"]


In [ ]:
###############################
# Прогон всех моделей и сохранение результатов
###############################

model_results = {}

for idx, exog_vars_initial in enumerate(exog_variants, 1):
    y, X, pooled_res, fe_res, re_res, pooled_success, fe_success, re_success = run_panel_regressions(
        df_reg, df_reg, dependent_var, exog_vars_initial, cov_type='dk', cluster_entity=True, df_name='df_reg'
    )

    model_results[f"Модель {idx} - OLS"] = pooled_res if pooled_success else None
    model_results[f"Модель {idx} - FE"] = fe_res if fe_success else None
    model_results[f"Модель {idx} - RE"] = re_res if re_success else None


In [ ]:
###############################
# Прогон всех моделей + тесты + экспорт
###############################

from Modules.panel_utils import collect_all_test_pvalues, run_spec_tests_robust
from Modules.model_results_export import ModelResultsAggregator, ensure_results_dir, add_model_set, build_and_export_with_tests
import os

model_spec_all = build_shock_variants(dependent_var, exog_vars_base, shock_vars)
exog_variants = model_spec_all["exog_variants"]

model_specs_all = []
tests_by_column = {}

for idx, exog_vars_initial in enumerate(exog_variants, 1):
    y, X, pooled_res, fe_res, re_res, pooled_success, fe_success, re_success = run_panel_regressions(
        df_reg, df_reg, dependent_var, exog_vars_initial, cov_type='dk', cluster_entity=True, df_name='df_reg'
    )

    spec_name = f"Модель {idx}"
    model_specs_all.append({
        'spec_name': spec_name,
        'dependent_var': dependent_var,
        'subsample': 'Общая выборка',
        'results': {
            'pooled': pooled_res if pooled_success else None,
            'fe': fe_res if fe_success else None,
            're': re_res if re_success else None
        }
    })

    tests = collect_all_test_pvalues(
        y, X, pooled_res, fe_res, re_res,
        pooled_success, fe_success, re_success,
        shock_vars=shock_vars
    )
    hausman_stat_r, hausman_pval_r, bp_lm_stat_r, bp_lm_pval_r, f_stat_r, f_pval_r = run_spec_tests_robust(
        y, X, pooled_res, fe_res, re_res,
        pooled_success, fe_success, re_success
    )
    spec_tests_robust = {
        "Hausman (FE vs RE) p-value (robust)": hausman_pval_r,
        "Breusch-Pagan LM (RE vs Pooled) p-value (robust)": bp_lm_pval_r,
        "F-test (FE vs Pooled) p-value (robust)": f_pval_r,
    }


    spec_tests = tests.get('spec_tests', {})
    spec_tests.update(spec_tests_robust)
    diag_tests = tests.get('diagnostics', {})

    for model_type in ['POOL', 'FE', 'RE']:
        col_name = f"{spec_name} ({model_type})"
        col_tests = {}
        for name, pval in spec_tests.items():
            col_tests[name] = pval
        for name, pval in diag_tests.get(model_type, {}).items():
            col_tests[name] = pval
        tests_by_column[col_name] = col_tests

aggregator_all = ModelResultsAggregator()
for spec in model_specs_all:
    add_model_set(aggregator_all, spec)

results_dir = ensure_results_dir('Results')
date_tag = pd.Timestamp.today().strftime('%d_%m_%y')
out_all = os.path.join(results_dir, f"all_models_with_tests_{date_tag}.xlsx")
build_and_export_with_tests(aggregator_all, tests_by_column, out_all, include_pvalues=True, decimals=3, test_decimals=6)
